In [10]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
os.environ["PINECONE_API_KEY"] = "pcsk_594Xgb_Qtwvagd2kRcdaJtLSdnXXrnn2RN81sRkGZyJYCqQmiEJY1iw6rDdNATRvSmfkCZ"
print("Pinecone API Key:", os.getenv("PINECONE_API_KEY"))
from Search_or_Discuss import classify_and_extract
from data_utils import fetch_papers, verify_results, chunk_papers
from Database import create_index, save_embeddings_to_index, delete_index
from retrieval import answer_query, clear_memory
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))




Pinecone API Key: pcsk_594Xgb_Qtwvagd2kRcdaJtLSdnXXrnn2RN81sRkGZyJYCqQmiEJY1iw6rDdNATRvSmfkCZ


In [16]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

def handleUserQuery(user_input, user_index):
    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)

    if classification["action"] == "search":
        filters = classification['filters']
        print("\nRunning Search with filters:", filters)

        # Step 1: Fetch papers
        papers = fetch_papers(filters)

        # Step 2: Verify & Filter papers
        filtered_papers = verify_results(papers, filters)

        # Step 3: Chunk papers
        paper_chunks = chunk_papers(filtered_papers)

        # Print sample chunks and save embeddings (update as needed)
        for i, chunk in enumerate(paper_chunks[:3]):
            print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")
            save_embeddings_to_index(index_name=user_index, text=[chunk.page_content])

        question = classification['question']
        answer = answer_query(question, index_obj, top_k=3)
        print("Generated Answer:")
        print(answer)

    elif classification["action"] == "discuss":
        question = classification['question']
        print("\nDiscussion Topic:", question)
        # Call the RAG pipeline to generate a discussion answer
        answer = answer_query(question, index_obj, top_k=3)
        print("Generated Answer:")
        print(answer)
    else:
        # Default behavior if classification doesn't match expected actions:
        print("Action not recognized. Please try again.")

# Use a loop to enable continuous conversation
while True:
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() in ["exit", "quit"]:
        break
    handleUserQuery(user_query, index_name)
delete_index(index_name=index_name)
clear_memory()

Index 'mief9cttsm' created successfully!

Running Search with filters: {'query': 'machine learning', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'newest'}

Chunk 1:
VideoMage: Multi-Subject and Motion Customization
of Text-to-Video Diffusion Models
Chi-Pin Huang1,†, Yen-Siang Wu2, Hung-Kai Chung2,
Kai-Po Chang1, Fu-En Yang3, and Yu-Chiang Frank Wang1,3,‡
1 Graduate Institute of Communication Engineering, National Taiwan University
2 National Taiwan University 3...
Upserted 1 embeddings into index 'mief9cttsm', namespace 'default'.

Chunk 2:
... on the grass.
<gnome> is walking <wolf> in the forest.
... in the park.
<gnome>
<wolf>
<gnome> is riding <wolf> in the beach.
... in the snow.
A lady is walking a dog. A person is riding a horse.
Figure 1. Illustration of multi-subject and motion customization.Given images of multiple subjects, ...
